# plot_trajectory — one run at a time

Interactive companion to `Analysis/plot_trajectory.py`. It imports the script rather than copying
it, so the figures here are identical to the ones the command line produces.

Hull area and wall contacts are read from the recording, both computed in Unity at capture time.
Nothing is recomputed here.

**Kernel:** `Python (PythonAnalysis)`.

In [ ]:
import sys
from pathlib import Path

# The scripts live one level up, in Analysis/. Make them importable from this notebook.
ANALYSIS = Path.cwd().parent
if str(ANALYSIS) not in sys.path:
    sys.path.insert(0, str(ANALYSIS))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# Point this at the batch you want to look at.
RECORDINGS = ANALYSIS.parent / "Assets/SimulationRecordings/Combinations/20260817_004421"
print("recordings:", RECORDINGS)
print("exists:", RECORDINGS.is_dir())

In [ ]:
import plot_trajectory as pt

files = sorted(p for p in RECORDINGS.glob("*.json")
               if p.stem != "batch_config" and not p.stem.endswith("_config"))

for i, p in enumerate(files):
    print(f"{i:3d}  {p.stem}")

## Pick one and load it

In [ ]:
RUN = files[0]          # change the index, or assign a Path directly

data = pt.load_trajectory(RUN)
times, areas, contacts, cumulative = pt.analyse(data)
header = data["header"]

print(f"{RUN.stem}")
print(f"   {header.get('swarmType')}, {header.get('agentCount')} agents, "
      f"{len(times)} frames over {times[-1]:.2f}s")
print(f"   hull {areas[0]:.1f} -> {areas[-1]:.1f} u²   ratio {areas[-1]/areas[0]:.2f}")
print(f"   contacts: peak {contacts.max()}, {cumulative[-1]} unique agents")
print(f"   ended: {header.get('endReason')}")

## The figure

In [ ]:
fig = pt.build_figure(RUN, data, times, areas, contacts, cumulative)
plt.show()

## The series as a table

Same four columns `--csv` writes. Useful for spotting where something changes.

In [ ]:
series = pd.DataFrame({
    "time_s": times,
    "hull_area": areas,
    "touching": contacts,
    "unique_touched": cumulative,
})

display(series.describe().round(2))
series.head(10)

## Where did the hull change fastest?

The steepest stretch is usually the interesting part of the clip.

In [ ]:
window = max(2, int(len(times) * 0.02))          # ~2% of the clip
rate = pd.Series(areas).diff(window) / pd.Series(times).diff(window)

peak = int(rate.abs().idxmax())
print(f"fastest change at t = {times[peak]:.2f}s, {rate[peak]:+.1f} u²/s")

fig, ax = plt.subplots(figsize=(10, 3.2), layout="constrained")
ax.plot(times, rate, linewidth=1.3, color="#1f77b4")
ax.axvline(times[peak], color="#d62728", linestyle="--", linewidth=1,
           label=f"peak {times[peak]:.2f}s")
ax.axhline(0, color="#888888", linewidth=0.8)
ax.set_xlabel("time (s)"); ax.set_ylabel("d(hull)/dt  (u²/s)")
ax.grid(alpha=0.25); ax.legend()
plt.show()

## Save this run's figure and CSV

In [ ]:
out_dir = RECORDINGS / "figures"
print(pt.plot(RUN, data, times, areas, contacts, cumulative, out_dir))
print(pt.write_csv(RUN, times, areas, contacts, cumulative, out_dir))

## Every run in the batch, inline

The equivalent of running the script over the folder, but rendered here.

In [ ]:
for p in files[:4]:                              # widen the slice for more
    d = pt.load_trajectory(p)
    t, a, c, u = pt.analyse(d)
    fig = pt.build_figure(p, d, t, a, c, u)
    plt.show()